In [ ]:
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from capstone.config import CATEGORICAL, FEATURES, NUMERIC, PLOT, STYLES, Col
from capstone.data import load_recipes
from capstone.features import gower_distance
from capstone.utilities import setup_logging

setup_logging("WARNING")

pd.set_option("display.precision", 2)
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.dpi"] = 110

In [ ]:
NUM, CAT, ALL = NUMERIC, CATEGORICAL, FEATURES
COLORS = PLOT.style_colors

df = load_recipes()
df[Col.style] = pd.Categorical(df[Col.style], categories=STYLES)
style = df[Col.style].astype(str)
df.head()

In [ ]:
df["style"].value_counts()

In [ ]:
df.groupby("style", observed=True)[NUM].agg(["mean", "median", "std", "min", "max"]).round(2)

In [ ]:
for col in CAT:
    display(pd.crosstab(df["style"], df[col]))

In [ ]:
fig, axes = plt.subplots(1, len(CAT), figsize=(17, 3.8))
for ax, col in zip(axes, CAT):
    share = pd.crosstab(df["style"], df[col], normalize="index")
    share.plot.bar(stacked=True, ax=ax, colormap="tab20", rot=0)
    ax.set_title(col)
    ax.set_xlabel("")
    ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1, 1))
fig.tight_layout()

In [ ]:
# Joint categorical profiles: how many distinct combinations, and which are shared across styles
profile = df[CAT].agg(" | ".join, axis=1).rename("profile")
pd.crosstab(profile, df["style"]).assign(total=lambda t: t.sum(axis=1)).sort_values("total", ascending=False)

In [ ]:
# Fat/sugar presence combined with type
df.assign(has_fat=df["fat"] > 0, has_sugar=df["sugar"] > 0).groupby("style", observed=True)[["has_fat", "has_sugar"]].mean()

In [ ]:
fig, axes = plt.subplots(1, len(NUM), figsize=(16, 3.5))
for ax, f in zip(axes, NUM):
    data = [df.loc[df["style"] == s, f] for s in STYLES]
    ax.boxplot(data, tick_labels=STYLES)
    for i, (s, d) in enumerate(zip(STYLES, data), start=1):
        ax.scatter(np.random.default_rng(0).normal(i, 0.05, len(d)), d, color=COLORS[s], alpha=0.7, zorder=3)
    ax.set_title(f)
    ax.tick_params(axis="x", rotation=30)
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, len(NUM), figsize=(16, 3))
for ax, f in zip(axes, NUM):
    bins = np.histogram_bin_edges(df[f], bins=12)
    for s in STYLES:
        ax.hist(df.loc[df["style"] == s, f], bins=bins, alpha=0.55, color=COLORS[s], label=s)
    ax.set_title(f)
axes[0].legend()
fig.tight_layout()

In [ ]:
# Numeric features split by categorical type (e.g. hydration by flour type, colored by style)
fig, axes = plt.subplots(len(CAT), 2, figsize=(13, 3.2 * len(CAT)))
rng = np.random.default_rng(0)
for row, col in zip(axes, CAT):
    cats = sorted(df[col].unique())
    for ax, num in zip(row, ["water", "yeast"]):
        for i, cat in enumerate(cats):
            sub = df[df[col] == cat]
            ax.scatter(rng.normal(i, 0.06, len(sub)), sub[num], c=sub["style"].astype(str).map(COLORS), alpha=0.8)
        ax.set_xticks(range(len(cats)), cats, rotation=20, fontsize=8)
        ax.set_ylabel(num)
        ax.set_title(f"{num} by {col}")
handles = [plt.Line2D([], [], marker="o", ls="", color=COLORS[s], label=s) for s in STYLES]
axes[0, 0].legend(handles=handles)
fig.tight_layout()

In [ ]:
pairs = list(combinations(NUM, 2))
fig, axes = plt.subplots(2, 5, figsize=(18, 6.5))
for ax, (x, y) in zip(axes.ravel(), pairs):
    for s in STYLES:
        sub = df[df["style"] == s]
        ax.scatter(sub[x], sub[y], color=COLORS[s], label=s, alpha=0.8)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
axes[0, 0].legend()
fig.tight_layout()

In [ ]:
# Gower distance over all 9 features: numeric range-normalized absolute difference, categorical 0/1 mismatch
D_all = gower_distance(df, NUM, CAT)
D_num = gower_distance(df, NUM, [])
D_cat = gower_distance(df, [], CAT)

def block_summary(D):
    out = pd.DataFrame(index=STYLES, columns=STYLES, dtype=float)
    for a in STYLES:
        for b in STYLES:
            block = D.loc[style == a, style == b].values
            vals = block[np.triu_indices_from(block, k=1)] if a == b else block.ravel()
            out.loc[a, b] = vals.mean()
    return out

for name, D in [("all features", D_all), ("numeric only", D_num), ("categorical only", D_cat)]:
    print(name)
    display(block_summary(D))

In [ ]:
order = df.sort_values("style").index
bounds = np.cumsum([(df["style"] == s).sum() for s in STYLES])[:-1] - 0.5
fig, axes = plt.subplots(1, 3, figsize=(20, 6.5))
for ax, (name, D) in zip(axes, [("all", D_all), ("numeric", D_num), ("categorical", D_cat)]):
    im = ax.imshow(D.loc[order, order], cmap="viridis_r", vmin=0, vmax=1)
    for b in bounds:
        ax.axhline(b, color="white", lw=2)
        ax.axvline(b, color="white", lw=2)
    ax.set_xticks(range(len(order)), order, rotation=90, fontsize=5)
    ax.set_yticks(range(len(order)), order, fontsize=5)
    ax.set_title(f"Gower distance - {name}")
fig.colorbar(im, ax=axes, shrink=0.7)

In [ ]:
# Mean distance of each recipe to every class (own class excludes self); nearest-class assignment
def mean_dist_to_classes(D):
    rows = {}
    for rid in D.index:
        r = {}
        for s in STYLES:
            members = D.index[(style == s) & (D.index != rid)]
            r[s] = D.loc[rid, members].mean()
        rows[rid] = r
    out = pd.DataFrame(rows).T
    out["nearest"] = out[STYLES].idxmin(axis=1)
    out["true"] = style
    return out

for name, D in [("all features", D_all), ("numeric only", D_num), ("categorical only", D_cat)]:
    res = mean_dist_to_classes(D)
    print(name, "- accuracy:", round((res["nearest"] == res["true"]).mean(), 3))
    display(pd.crosstab(res["true"], res["nearest"]))
res_all = mean_dist_to_classes(D_all)
res_all.loc[res_all["nearest"] != res_all["true"]]

In [ ]:
# Nearest neighbor of each recipe (all features) and its class
D_nn = D_all.where(~np.eye(len(D_all), dtype=bool), np.inf)
nn = pd.DataFrame({"nn": D_nn.idxmin(), "nn_dist": D_nn.min()})
nn["true"] = style
nn["nn_style"] = style.loc[nn["nn"]].values
display(pd.crosstab(nn["true"], nn["nn_style"]))
nn.loc[nn["true"] != nn["nn_style"]]

In [ ]:
# Numeric overlap: Cohen's d, range overlap, and fraction of recipes inside the overlap range per style pair
rows = []
for f in NUM:
    for a, b in combinations(STYLES, 2):
        x, y = df.loc[df["style"] == a, f], df.loc[df["style"] == b, f]
        lo, hi = max(x.min(), y.min()), min(x.max(), y.max())
        pooled = np.sqrt(((len(x) - 1) * x.var() + (len(y) - 1) * y.var()) / (len(x) + len(y) - 2))
        in_overlap = (x.between(lo, hi).sum() + y.between(lo, hi).sum()) / (len(x) + len(y)) if lo <= hi else 0.0
        rows.append({"feature": f, "pair": f"{a} vs {b}", "cohens_d": (x.mean() - y.mean()) / pooled if pooled else np.nan,
                     "frac_in_overlap": in_overlap})
sep_num = pd.DataFrame(rows)
display(sep_num.pivot(index="feature", columns="pair", values="cohens_d"))
sep_num.pivot(index="feature", columns="pair", values="frac_in_overlap")

In [ ]:
# Categorical overlap: total-variation distance between category distributions per style pair (0 = identical, 1 = disjoint)
rows = []
for f in CAT:
    share = pd.crosstab(df["style"], df[f], normalize="index")
    for a, b in combinations(STYLES, 2):
        rows.append({"feature": f, "pair": f"{a} vs {b}", "tv_distance": 0.5 * (share.loc[a] - share.loc[b]).abs().sum()})
pd.DataFrame(rows).pivot(index="feature", columns="pair", values="tv_distance")

In [ ]:
# Distinguishing numeric features per class: one-vs-rest Cohen's d and rank-based AUC
def auc(pos, neg):
    ranks = pd.concat([pos, neg]).rank()
    return (ranks.iloc[: len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg))

rows = []
for s in STYLES:
    in_, out = df[df["style"] == s], df[df["style"] != s]
    for f in NUM:
        pooled = np.sqrt(((len(in_) - 1) * in_[f].var() + (len(out) - 1) * out[f].var()) / (len(df) - 2))
        rows.append({"style": s, "feature": f, "mean_in": in_[f].mean(), "mean_out": out[f].mean(),
                     "cohens_d": (in_[f].mean() - out[f].mean()) / pooled if pooled else np.nan, "auc": auc(in_[f], out[f])})
ovr_num = pd.DataFrame(rows)
ovr_num.assign(abs_d=ovr_num["cohens_d"].abs()).sort_values(["style", "abs_d"], ascending=[True, False]).drop(columns="abs_d")

In [ ]:
# Distinguishing categories per class: share inside vs outside the class, and lift
rows = []
for s in STYLES:
    in_, out = df[df["style"] == s], df[df["style"] != s]
    for f in CAT:
        for cat in sorted(df[f].unique()):
            p_in, p_out = (in_[f] == cat).mean(), (out[f] == cat).mean()
            rows.append({"style": s, "feature": f, "category": cat, "share_in": p_in, "share_out": p_out,
                         "diff": p_in - p_out, "lift": p_in / (df[f] == cat).mean()})
ovr_cat = pd.DataFrame(rows)
ovr_cat.assign(abs_diff=ovr_cat["diff"].abs()).sort_values(["style", "abs_diff"], ascending=[True, False]).groupby("style").head(8).drop(columns="abs_diff")

In [ ]:
# Effect-size summary on a comparable 0-1 scale: numeric = |AUC - 0.5| * 2, categorical = Cramer's V of feature vs (style == s)
def cramers_v(a, b):
    t = pd.crosstab(a, b)
    obs = t.values.astype(float)
    exp = obs.sum(1, keepdims=True) * obs.sum(0, keepdims=True) / obs.sum()
    chi2 = ((obs - exp) ** 2 / exp).sum()
    return np.sqrt(chi2 / (obs.sum() * (min(obs.shape) - 1)))

summary = pd.DataFrame(index=ALL, columns=STYLES, dtype=float)
for s in STYLES:
    for f in NUM:
        summary.loc[f, s] = abs(ovr_num.query("style == @s and feature == @f")["auc"].iloc[0] - 0.5) * 2
    for f in CAT:
        summary.loc[f, s] = cramers_v(df[f], style == s)
summary["overall"] = [
    cramers_v(df[f], style) if f in CAT else np.nan for f in ALL
]
display(summary)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(summary[STYLES].astype(float), cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(3), STYLES)
ax.set_yticks(range(len(ALL)), ALL)
for i in range(len(ALL)):
    for j in range(3):
        ax.text(j, i, f"{summary.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im)
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharex=True)
for ax, s in zip(axes, STYLES):
    sub = ovr_num[ovr_num["style"] == s].set_index("feature")["cohens_d"].reindex(NUM)
    ax.barh(NUM, sub, color=[COLORS[s] if v > 0 else "lightgray" for v in sub])
    ax.axvline(0, color="k", lw=0.8)
    ax.set_title(f"{s} vs rest")
    ax.set_xlabel("Cohen's d")
fig.tight_layout()

In [ ]:
# Pairwise: best single-feature rule per style pair (balanced accuracy; numeric = best threshold, categorical = majority-class lookup)
def best_threshold_acc(x, y):
    vals = np.sort(np.unique(np.concatenate([x, y])))
    best = 0.5
    for t in (vals[:-1] + vals[1:]) / 2:
        best = max(best, ((x < t).mean() + (y >= t).mean()) / 2, ((x >= t).mean() + (y < t).mean()) / 2)
    return best

def lookup_acc(x, y):
    return sum(max((x == k).sum() / len(x), (y == k).sum() / len(y)) for k in set(x) | set(y)) / 2

rows = []
for a, b in combinations(STYLES, 2):
    for f in ALL:
        x, y = df.loc[df["style"] == a, f].values, df.loc[df["style"] == b, f].values
        rows.append({"pair": f"{a} vs {b}", "feature": f, "balanced_acc": best_threshold_acc(x, y) if f in NUM else lookup_acc(x, y)})
pd.DataFrame(rows).pivot(index="feature", columns="pair", values="balanced_acc").loc[ALL]

In [ ]:
# PCA via SVD on standardized numeric features + one-hot categorical features
X = pd.concat([df[NUM], pd.get_dummies(df[CAT], dtype=float)], axis=1)
Xz = (X - X.mean()) / X.std()
Xz = Xz.loc[:, X.std() > 0]
U, S, Vt = np.linalg.svd(Xz.values, full_matrices=False)
pcs = pd.DataFrame(Xz.values @ Vt.T[:, :3], index=Xz.index, columns=["PC1", "PC2", "PC3"])
explained = pd.Series(S**2 / (S**2).sum(), index=[f"PC{i+1}" for i in range(len(S))])
loadings = pd.DataFrame(Vt.T[:, :3], index=Xz.columns, columns=["PC1", "PC2", "PC3"])
display(explained.head(6).to_frame("explained_variance"))
display(loadings.reindex(loadings["PC1"].abs().sort_values(ascending=False).index).head(12))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (px, py) in zip(axes, [("PC1", "PC2"), ("PC1", "PC3")]):
    for s in STYLES:
        m = style == s
        ax.scatter(pcs.loc[m, px], pcs.loc[m, py], color=COLORS[s], label=s, alpha=0.85)
    ax.set_xlabel(f"{px} ({explained[px]:.0%})")
    ax.set_ylabel(f"{py} ({explained[py]:.0%})")
axes[0].legend()
fig.tight_layout()

In [ ]:
# Metric MDS (classical) on Gower distance over all features
n = len(D_all)
J = np.eye(n) - np.ones((n, n)) / n
B = -0.5 * J @ (D_all.values ** 2) @ J
w, V = np.linalg.eigh(B)
idx = np.argsort(w)[::-1][:2]
coords = V[:, idx] * np.sqrt(w[idx])

fig, ax = plt.subplots(figsize=(6.5, 5.5))
for s in STYLES:
    m = (style == s).values
    ax.scatter(coords[m, 0], coords[m, 1], color=COLORS[s], label=s, alpha=0.85)
ax.set_title("Classical MDS - Gower distance (all features)")
ax.legend()
fig.tight_layout()

In [ ]:
# Mixed association matrix: |Pearson| (num-num), Cramer's V (cat-cat), correlation ratio eta (cat-num)
def eta(cat, num):
    groups = [num[cat == k] for k in cat.unique()]
    ss_between = sum(len(g) * (g.mean() - num.mean()) ** 2 for g in groups)
    ss_total = ((num - num.mean()) ** 2).sum()
    return np.sqrt(ss_between / ss_total) if ss_total else np.nan

assoc = pd.DataFrame(index=ALL, columns=ALL, dtype=float)
for a in ALL:
    for b in ALL:
        if a in NUM and b in NUM:
            assoc.loc[a, b] = abs(df[a].corr(df[b]))
        elif a in CAT and b in CAT:
            assoc.loc[a, b] = 1.0 if a == b else cramers_v(df[a], df[b])
        elif a in CAT:
            assoc.loc[a, b] = eta(df[a], df[b])
        else:
            assoc.loc[a, b] = eta(df[b], df[a])

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(assoc, vmin=0, vmax=1, cmap="Blues")
ax.set_xticks(range(len(ALL)), ALL, rotation=45, ha="right")
ax.set_yticks(range(len(ALL)), ALL)
for i in range(len(ALL)):
    for j in range(len(ALL)):
        ax.text(j, i, f"{assoc.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im)
fig.tight_layout()

In [ ]:
# Within-class outliers: mean Gower distance to own class members
own = pd.Series({rid: D_all.loc[rid, D_all.index[(style == style[rid]) & (D_all.index != rid)]].mean() for rid in D_all.index})
outliers = pd.DataFrame({"style": style, "mean_dist_own_class": own, "mean_dist_numeric": pd.Series({rid: D_num.loc[rid, D_num.index[(style == style[rid]) & (D_num.index != rid)]].mean() for rid in D_num.index}),
                         "mean_dist_categorical": pd.Series({rid: D_cat.loc[rid, D_cat.index[(style == style[rid]) & (D_cat.index != rid)]].mean() for rid in D_cat.index})})
outliers.sort_values("mean_dist_own_class", ascending=False).head(10)

In [ ]:
df.groupby(df["source"].str.extract(r"https?://(?:www\\.)?([^/]+)")[0].rename("domain"))["style"].value_counts().unstack(fill_value=0)